# Stage 08 — Spare Parts EDA: Demand Characterisation
**Dashboard page:** Inventory Analysis > Spare Parts EDA tab
**Key output:** CV / p_zero quadrant map that determines forecast tier assignment

**Demand taxonomy:**
| CV | p_zero | Category | Forecast tier |
|---|---|---|---|
| ≥0.5 | ≥0.7 | Lumpy/Erratic | Tier 2: Croston |
| <0.5 | ≥0.7 | Intermittent  | Tier 2: Croston |
| <0.5 | <0.5 | Smooth        | Tier 3/4: ETS or LightGBM |
| active_months=0 | — | Non-moving | Tier 0: zero forecast |

In [ ]:
import sys, warnings, re
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INTERIM   = PROJECT_ROOT / "data" / "interim"
PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS   = PROJECT_ROOT / "data" / "outputs"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
})
PALETTE = ["#4361EE","#EF4444","#2CC56F","#F59E0B","#A855F7",
           "#64748B","#06B6D4","#F97316","#10B981","#8B5CF6"]
STATUS_COLORS = {"stockout":"#EF4444","critical":"#F97316",
                 "low":"#F59E0B","ok":"#2CC56F","excess":"#4361EE"}
TIER_COLORS   = {"critical":"#EF4444","managed":"#F97316",
                 "watch":"#F59E0B","rationalise":"#94A3B8"}

def load(name, base=None):
    if base:
        p = Path(base) / name
        if p.exists(): return pd.read_parquet(p)
    for b in [INTERIM, PROCESSED, OUTPUTS]:
        p = b / name
        if p.exists(): return pd.read_parquet(p)
    raise FileNotFoundError(f"{name} not found")

def fmt_lkr(v):
    if abs(v) >= 1e9: return f"LKR {v/1e9:.1f}B"
    if abs(v) >= 1e6: return f"LKR {v/1e6:.1f}M"
    return f"LKR {v:,.0f}"


In [ ]:
feat = load("spare_parts_features.parquet")
print(f"Spare parts features: {len(feat):,} SKUs")
print(feat.describe()[["active_months","p_zero","avg_monthly_demand","cv","total_issue_value_lkr"]].round(3).to_string())


In [ ]:
fig,axes = plt.subplots(2,2,figsize=(14,10))

# CV histogram
cv_c = feat["cv"].clip(upper=5)
axes[0,0].hist(cv_c.dropna(),bins=70,color=PALETTE[0],edgecolor="white",alpha=0.8)
axes[0,0].axvline(0.5,color=PALETTE[1],ls="--",lw=1.8,label="X|Y  CV=0.5")
axes[0,0].axvline(1.0,color=PALETTE[2],ls="--",lw=1.8,label="Y|Z  CV=1.0")
axes[0,0].set_title("CV Distribution  (X<0.5 predictable, Y 0.5-1.0, Z≥1.0 erratic)")
axes[0,0].set_xlabel("CV (clipped at 5)"); axes[0,0].legend(fontsize=8)

# p_zero histogram
axes[0,1].hist(feat["p_zero"].dropna(),bins=50,color=PALETTE[3],edgecolor="white",alpha=0.8)
axes[0,1].axvline(0.7,color=PALETTE[1],ls="--",lw=1.8,label="Intermittency threshold 0.70")
pct = (feat["p_zero"]>=0.7).mean()*100
axes[0,1].set_title(f"p_zero Distribution — {pct:.0f}% of SKUs intermittent (p_zero≥0.70)")
axes[0,1].set_xlabel("Fraction of zero-demand months"); axes[0,1].legend(fontsize=8)

# Active months
axes[1,0].hist(feat["active_months"].dropna(),bins=50,color=PALETTE[4],edgecolor="white",alpha=0.8)
for thresh,lbl,col in [(1,"Tier1→2",PALETTE[1]),(3,"Tier2→3","#059669"),(12,"NHITS eligible","#DC2626")]:
    axes[1,0].axvline(thresh,color=col,ls="--",alpha=0.8,label=f"{lbl} ({thresh}m)")
axes[1,0].set_title("Active Months per SKU (months with qty>0)")
axes[1,0].set_xlabel("Active months"); axes[1,0].legend(fontsize=7)

# CV vs p_zero map
dc_col = "demand_category" if "demand_category" in feat.columns else None
samp = feat.sample(min(2000,len(feat)),random_state=42)
if dc_col:
    for i,cat in enumerate(sorted(samp[dc_col].dropna().unique())):
        m = samp[dc_col]==cat
        axes[1,1].scatter(samp.loc[m,"p_zero"],samp.loc[m,"cv"].clip(upper=4),
                          alpha=0.4,s=18,color=PALETTE[i%len(PALETTE)],label=cat)
    axes[1,1].legend(fontsize=8,markerscale=2,title="Demand category")
else:
    axes[1,1].scatter(samp["p_zero"],samp["cv"].clip(upper=4),alpha=0.3,s=15,color=PALETTE[0])
axes[1,1].axvline(0.7,color="gray",ls="--",alpha=0.6,lw=1)
axes[1,1].axhline(0.5,color="gray",ls="--",alpha=0.6,lw=1)
for px,py,txt in [(0.05,3.7,"Erratic"),(0.75,3.7,"Lumpy"),(0.05,0.2,"Smooth"),(0.75,0.2,"Intermittent")]:
    axes[1,1].text(px,py,txt,fontsize=9,color="gray",alpha=0.8)
axes[1,1].set_title("Demand Characterisation Map: CV vs p_zero")
axes[1,1].set_xlabel("p_zero"); axes[1,1].set_ylabel("CV (clipped at 4)")
plt.tight_layout(); plt.show()


In [ ]:
if "demand_category" in feat.columns:
    dc = feat["demand_category"].value_counts()
    print("Demand category distribution:")
    for cat,cnt in dc.items(): print(f"  {cat:25s}: {cnt:5,} ({cnt/len(feat)*100:.1f}%)")

zero_sku = (feat["active_months"]==0).sum()
print(f"
Non-moving SKUs (zero history): {zero_sku:,} ({zero_sku/len(feat)*100:.1f}%)")

# Top SKUs by issue value
top = feat.nlargest(15,"total_issue_value_lkr")
fig,ax = plt.subplots(figsize=(12,5))
top["total_issue_value_lkr"].sort_values().plot(kind="barh",ax=ax,color=PALETTE[0],edgecolor="white")
ax.set_title("Top 15 SKUs by Total Historical Issue Value (LKR)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{x/1e6:.1f}M"))
ax.set_yticklabels([str(d)[:45] for d in top.sort_values("total_issue_value_lkr")["description"]],fontsize=7)
plt.tight_layout(); plt.show()
